# GéoMarketing IDF : J10d — Santé et action sociale en Île-de-France

## Objectifs
- Extraire les observations du domaine santé et action sociale de la BPE.
- Conserver les données détaillées pour la cartographie.
- Calculer des comptages, densités et parts par commune.
- Enrichir le profil J10c sans modifier ses indicateurs.

## Unité de comptage
Une observation BPE : établissement, service ou activité professionnelle
selon le type d'équipement.

## Limites
- Pas de décompte de personnes ou de bâtiments uniques.
- Pas de mesure du nombre de patients ni de la fréquentation.
- Pas de mesure des délais de rendez-vous ou de la disponibilité des soins.
- Les capacités ne sont pas additionnées entre catégories.
- Les coordonnées nécessiteront une validation géographique au J11.

In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("Pandas :", pd.__version__)

Pandas : 2.2.2


In [2]:
#Définir les chemins

RACINE = Path(
    r"C:\Users\almou\OneDrive\GeoMarketing_IDF"
)

DOSSIER_INTERIM = RACINE / "data" / "interim" / "j10d"
DOSSIER_PROCESSED = RACINE / "data" / "processed"

FICHIER_BPE = (
    RACINE / "data" / "raw" / "bpe" / "BPE25.csv"
)

FICHIER_PROFIL = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10c.csv"
)

FICHIER_METADATA_J10C = (
    RACINE / "data" / "interim" / "j10c" / "metadata_j10c.json"
)

FICHIER_DETAIL = (
    DOSSIER_PROCESSED / "sante_action_sociale_idf_j10d.csv"
)

FICHIER_COMMUNES = (
    DOSSIER_INTERIM / "sante_action_sociale_communes.csv"
)

FICHIER_SORTIE = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10d.csv"
)

for fichier in [FICHIER_BPE, FICHIER_PROFIL]:
    if not fichier.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {fichier}")

DOSSIER_INTERIM.mkdir(parents=True, exist_ok=True)
DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Profil d'entrée :", FICHIER_PROFIL)
print("Profil de sortie :", FICHIER_SORTIE)

Profil d'entrée : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10c.csv
Profil de sortie : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10d.csv


In [3]:
#Fonctions utiles

def normaliser_nom_colonne(nom):
    nom = unicodedata.normalize(
        "NFKD", str(nom).strip().upper()
    )
    nom = "".join(
        caractere
        for caractere in nom
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"[^A-Z0-9]+", "_", nom).strip("_")


def nettoyer_texte(serie):
    resultat = serie.astype("string").str.strip()

    manquants = {
        "", "NA", "NAN", "NONE", "NULL",
        "<NA>", "_U", "_Z", "[ND]",
    }

    return resultat.mask(
        resultat.str.upper().isin(manquants)
    )


def normaliser_code_commune(serie):
    resultat = nettoyer_texte(serie).str.upper()
    resultat = resultat.str.replace(r"\.0$", "", regex=True)

    masque = resultat.str.fullmatch(r"\d{1,5}", na=False)

    return resultat.where(
        ~masque,
        resultat.str.zfill(5),
    )


def convertir_nombre(serie):
    return pd.to_numeric(
        nettoyer_texte(serie)
        .str.replace("\u202f", "", regex=False)
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def sauvegarder_csv(table, chemin):
    table.to_csv(
        chemin,
        index=False,
        encoding="utf-8-sig",
    )


def sha256_fichier(chemin):
    empreinte = hashlib.sha256()

    with open(chemin, "rb") as flux:
        for bloc in iter(lambda: flux.read(1024 * 1024), b""):
            empreinte.update(bloc)

    return empreinte.hexdigest()

In [4]:
#Charger le profil J10c

profil = pd.read_csv(
    FICHIER_PROFIL,
    dtype="string",
    encoding="utf-8-sig",
)

profil.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil.columns
]

if profil.columns.duplicated().any():
    raise ValueError("Colonnes dupliquées dans le profil.")

if "CODGEO" not in profil.columns:
    raise ValueError("La colonne CODGEO est absente.")

profil["CODGEO"] = normaliser_code_commune(profil["CODGEO"])

if not profil["CODGEO"].str.fullmatch(r"\d{5}", na=False).all():
    raise ValueError("Codes communaux manquants ou invalides.")

if not profil["CODGEO"].is_unique:
    raise ValueError("Plusieurs lignes existent pour une commune.")

DEPARTEMENTS_IDF = {
    "75", "77", "78", "91", "92", "93", "94", "95"
}

if not profil["CODGEO"].str[:2].isin(DEPARTEMENTS_IDF).all():
    raise ValueError("Codes hors Île-de-France dans le profil.")

colonne_temoin = "NB_OBS_SERVICES_HORS_RESTAURATION_BPE2025"

if colonne_temoin not in profil.columns:
    raise ValueError("Le profil ne contient pas le total attendu du J10c.")

codes_profil = set(profil["CODGEO"])

print("Communes :", len(profil))
print("Colonnes initiales :", len(profil.columns))
display(profil.head())

Communes : 1266
Colonnes initiales : 596


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,...,DENSITE_OBS_GENDARMERIE_10000_HAB,DENSITE_OBS_COURS_APPEL_10000_HAB,DENSITE_OBS_PRUDHOMMES_10000_HAB,DENSITE_OBS_TRIBUNAUX_COMMERCE_10000_HAB,DENSITE_OBS_FINANCES_PUBLIQUES_REGIONALES_10000_HAB,DENSITE_OBS_FINANCES_PUBLIQUES_DEPARTEMENTALES_10000_HAB,DENSITE_OBS_FRANCE_TRAVAIL_10000_HAB,DENSITE_OBS_MAISONS_JUSTICE_DROIT_10000_HAB,DENSITE_OBS_ANTENNES_JUSTICE_10000_HAB,DENSITE_OBS_CONSEILS_ACCES_DROIT_10000_HAB,DENSITE_OBS_FRANCE_SERVICES_10000_HAB,DENSITE_OBS_MAIRIES_10000_HAB,DENSITE_OBS_BUREAUX_AIDE_JURIDICTIONNELLE_10000_HAB,DENSITE_OBS_TRIBUNAUX_JUDICIAIRES_10000_HAB,DENSITE_OBS_TRIBUNAUX_PROXIMITE_10000_HAB,DENSITE_OBS_DECHETERIES_10000_HAB,DENSITE_OBS_COURS_CRIMINELLES_DEPARTEMENTALES_10000_HAB,DENSITE_OBS_COURS_ASSISES_10000_HAB,DENSITE_OBS_JURIDICTIONS_ADMINISTRATIVES_10000_HAB,DENSITE_OBS_TRIBUNAUX_CORRECTIONNELS_10000_HAB,DENSITE_OBS_TRIBUNAUX_POLICE_10000_HAB,DENSITE_OBS_TRIBUNAUX_ENFANTS_10000_HAB,DENSITE_OBS_POLICE_10000_HAB,DENSITE_OBS_BANQUES_10000_HAB,DENSITE_OBS_FUNERAIRE_10000_HAB,DENSITE_OBS_POSTE_BUREAUX_10000_HAB,DENSITE_OBS_POSTE_RELAIS_10000_HAB,DENSITE_OBS_POSTE_AGENCES_10000_HAB,DENSITE_OBS_REPARATION_VEHICULES_MATERIEL_AGRICOLE_10000_HAB,DENSITE_OBS_CONTROLE_VEHICULES_10000_HAB,DENSITE_OBS_LOCATION_VEHICULES_10000_HAB,DENSITE_OBS_FORMATION_CONDUITE_10000_HAB,DENSITE_OBS_MACONNERIE_10000_HAB,DENSITE_OBS_PLATRERIE_PEINTURE_10000_HAB,DENSITE_OBS_MENUISERIE_CHARPENTE_SERRURERIE_10000_HAB,DENSITE_OBS_PLOMBERIE_COUVERTURE_CHAUFFAGE_10000_HAB,DENSITE_OBS_ELECTRICITE_10000_HAB,DENSITE_OBS_ENTREPRISES_BATIMENT_10000_HAB,DENSITE_OBS_COIFFURE_10000_HAB,DENSITE_OBS_VETERINAIRES_10000_HAB,DENSITE_OBS_INTERIM_10000_HAB,DENSITE_OBS_IMMOBILIER_10000_HAB,DENSITE_OBS_ENTRETIEN_LINGE_10000_HAB,DENSITE_OBS_BEAUTE_ONGLES_10000_HAB,DENSITE_OBS_SERVICES_PUBLICS_10000_HAB,DENSITE_OBS_BANQUES_POSTE_FUNERAIRE_10000_HAB,DENSITE_OBS_AUTOMOBILE_CONDUITE_10000_HAB,DENSITE_OBS_ARTISANS_BATIMENT_10000_HAB,DENSITE_OBS_SERVICES_PERSONNELS_IMMOBILIERS_EMPLOI_10000_HAB,DENSITE_OBS_SERVICES_HORS_RESTAURATION_10000_HAB
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508206,513386.75602884,457511.658563684,387099.215350194,302819.494109117,154040.332161959,24566.9822779995,787667.317537046,970898.414592524,481426.8085490755,178607.31443995849,12.98,24.29,21.65,37.26,45.93,22.78,8.45,-76622.0,-3.5,-0.59,33650.0,16.8,1551133.75022,188002.0,73.38459010221389,88.94429449710343,20911,8329,12553,29,0,98.93055085738077,39.40474191053151,59.38860910108081,39.8307111089857,60.03060590120033,200.61215141394868,54155.23178,51427.58154,87388.40827,75287.99165,59522.89891,...,0.004731029164429284,0.004731029164429284,0.004731029164429284,0.004731029164429284,0.10881367078187354,0.0,0.08515852495972712,0.014193087493287852,0.0,0.004731029164429284,0.056772349973151406,0.09462058328858569,0.004731029164429284,0.004731029164429284,0.0,0.047310291644292846,0.004731029164429284,0.004731029164429284,0.00946205832885856

In [5]:
#Définir les groupes et catégories

GROUPES = {
    "D1": "STRUCTURES_SOINS",
    "D2": "PROFESSIONNELS_SANTE_RECENSES",
    "D3": "PHARMACIES_SERVICES_SANITAIRES",
    "D4": "SERVICES_PERSONNES_AGEES",
    "D5": "ENFANCE_FAMILLES_VIE_SOCIALE",
    "D6": "ACCOMPAGNEMENT_HANDICAP",
    "D7": "PROTECTION_HEBERGEMENT_SOCIAL",
}

TYPES = {
    # Structures de soins
    "D101": "SOINS_COURTE_DUREE",
    "D102": "SOINS_SUITE_READAPTATION",
    "D103": "SOINS_LONGUE_DUREE",
    "D104": "ETABLISSEMENTS_PSYCHIATRIQUES",
    "D105": "CENTRES_CANCEROLOGIE",
    "D106": "SERVICES_URGENCES",
    "D107": "MATERNITES",
    "D108": "CENTRES_SANTE",
    "D109": "PSYCHIATRIE_AMBULATOIRE",
    "D110": "CENTRES_PREVENTION_MEDICALE",
    "D111": "SERVICES_DIALYSE",
    "D112": "HOSPITALISATION_DOMICILE",
    "D113": "MAISONS_SANTE_PLURIDISCIPLINAIRES",
    "D114": "SERVICES_ADDICTOLOGIE",
    "D115": "SANTE_MATERNELLE_INFANTILE",

    # Professionnels recensés
    "D245": "PROFESSIONNELS_APPAREILLAGE",
    "D246": "AUDIOPROTHESISTES",
    "D247": "ERGOTHERAPEUTES",
    "D248": "PSYCHOMOTRICIENS",
    "D249": "DIETETICIENS",
    "D250": "PSYCHOLOGUES",
    "D251": "ALLERGOLOGUES",
    "D252": "ANESTHESISTES_REANIMATEURS",
    "D253": "CHIRURGIE_GENERALE",
    "D254": "CHIRURGIE_ORTHOPEDIQUE_PLASTIQUE_TRAUMATOLOGIQUE",
    "D255": "ENDOCRINOLOGUES",
    "D256": "GERIATRES",
    "D257": "HEMATOLOGUES",
    "D258": "MEDECINE_PHYSIQUE_READAPTATION",
    "D259": "NEUROLOGIE",
    "D260": "ONCOLOGIE_ANATOMIE_CYTOLOGIE_PATHOLOGIQUES",
    "D261": "RHUMATOLOGUES",
    "D262": "UROLOGIE_NEPHROLOGIE",
    "D265": "MEDECINS_GENERALISTES",
    "D266": "CARDIOLOGIE",
    "D267": "DERMATOLOGIE_VENEREOLOGIE",
    "D268": "GASTRO_ENTEROLOGIE_HEPATOLOGIE",
    "D269": "PSYCHIATRIE",
    "D270": "OPHTALMOLOGIE",
    "D271": "OTO_RHINO_LARYNGOLOGIE",
    "D272": "PEDIATRIE",
    "D273": "PNEUMOLOGIE",
    "D274": "RADIODIAGNOSTIC_IMAGERIE_MEDICALE",
    "D275": "STOMATOLOGIE",
    "D276": "GYNECOLOGIE_OBSTETRIQUE",
    "D277": "CHIRURGIENS_DENTISTES",
    "D278": "SAGES_FEMMES",
    "D279": "MASSEURS_KINESITHERAPEUTES",
    "D280": "PEDICURES_PODOLOGUES",
    "D281": "INFIRMIERS",
    "D282": "ORTHOPHONISTES",
    "D283": "ORTHOPTISTES",

    # Offre sanitaire complémentaire
    "D302": "LABORATOIRES_BIOLOGIE_MEDICALE",
    "D303": "AMBULANCES",
    "D304": "TRANSFUSION_SANGUINE",
    "D305": "ETABLISSEMENTS_THERMAUX",
    "D307": "PHARMACIES",

    # Personnes âgées
    "D401": "HEBERGEMENT_PERSONNES_AGEES",
    "D402": "SOINS_DOMICILE_PERSONNES_AGEES",
    "D403": "AIDE_PERSONNES_AGEES",

    # Enfance, familles et vie sociale
    "D502": "ACCUEIL_JEUNES_ENFANTS",
    "D503": "LIEUX_ACCUEIL_ENFANTS_PARENTS",
    "D504": "RELAIS_PETITE_ENFANCE",
    "D505": "ACCUEILS_LOISIRS_SANS_HEBERGEMENT",
    "D506": "CENTRES_SOCIAUX",
    "D507": "MEDIATION_FAMILIALE",
    "D508": "ESPACES_VIE_SOCIALE",
    "D509": "MICRO_CRECHES_FINANCEMENT_PAJE",

    # Handicap
    "D601": "HEBERGEMENT_ENFANTS_HANDICAP",
    "D602": "SERVICES_ENFANTS_HANDICAP",
    "D603": "ACCUEIL_HEBERGEMENT_ADULTES_HANDICAP",
    "D604": "AIDE_ADULTES_HANDICAP",
    "D605": "TRAVAIL_PROTEGE",
    "D606": "SOINS_DOMICILE_ADULTES_HANDICAP",
    "D607": "AIDE_HANDICAP_VISUEL_AUDITIF",

    # Protection et hébergement social
    "D701": "HEBERGEMENT_PROTECTION_ENFANCE",
    "D702": "ACTION_EDUCATIVE_PROTECTION_ENFANCE",
    "D703": "HEBERGEMENT_REINSERTION_SOCIALE",
    "D704": "HEBERGEMENT_PROVISOIRE",
    "D705": "ACCUEIL_DEMANDEURS_ASILE",
    "D710": "AUTRES_HEBERGEMENTS_PUBLICS_DIFFICULTE",
    "D711": "AIDE_FEMMES_VULNERABILITE",
}

assert len(TYPES.values()) == len(set(TYPES.values()))
assert not (set(TYPES.values()) & set(GROUPES.values()))

referentiel = pd.DataFrame({
    "TYPEQU": list(TYPES),
    "CATEGORIE": list(TYPES.values()),
})

referentiel["GROUPE_CODE"] = referentiel["TYPEQU"].str[:2]
referentiel["GROUPE"] = referentiel["GROUPE_CODE"].map(GROUPES)

referentiel["LIMITE_SPECIFIQUE"] = ""

limites_specifiques = {
    "D246": "Champ incluant certains professionnels salariés du privé.",
    "D282": "Champ limité aux moins de 62 ans selon la nomenclature 2025.",
    "D283": "Champ limité aux moins de 62 ans selon la nomenclature 2025.",
    "D401": "Ne correspond pas uniquement aux EHPAD.",
    "D502": "Peut inclure des micro-crèches ; champ distinct de D509.",
    "D509": "Micro-crèches du champ de financement PAJE.",
    "D710": "Catégorie non exhaustive.",
}

referentiel["LIMITE_SPECIFIQUE"] = (
    referentiel["TYPEQU"].map(limites_specifiques).fillna("")
)

sauvegarder_csv(
    referentiel,
    DOSSIER_INTERIM / "referentiel_sante_action_sociale.csv",
)

display(referentiel)

,TYPEQU,CATEGORIE,GROUPE_CODE,GROUPE,LIMITE_SPECIFIQUE
0,D101,SOINS_COURTE_DUREE,D1,STRUCTURES_SOINS,
1,D102,SOINS_SUITE_READAPTATION,D1,STRUCTURES_SOINS,
2,D103,SOINS_LONGUE_DUREE,D1,STRUCTURES_SOINS,
3,D104,ETABLISSEMENTS_PSYCHIATRIQUES,D1,STRUCTURES_SOINS,
4,D105,CENTRES_CANCEROLOGIE,D1,STRUCTURES_SOINS,
...,...,...,...,...,...
77,D703,HEBERGEMENT_REINSERTION_SOCIALE,D7,PROTECTION_HEBERGEMENT_SOCIAL,
78,D704,HEBERGEMENT_PROVISOIRE,D7,PROTECTION_HEBERGEMENT_SOCIAL,
79,D705,ACCUEIL_DEMANDEURS_ASILE,D7,PROTECTION_HEBERGEMENT_SOCIAL,
80,D710,AUTRES_HEBERGEMENTS_PUBLICS_DIFFICULTE,D7,PROTECTION_HEBERGEMENT_SOCIAL,Catégorie non exhaustive.


In [6]:
#Vérifier l'en-tête BPE

with open(
    FICHIER_BPE,
    "r",
    encoding="utf-8-sig",
) as flux:
    premiere_ligne = flux.readline()

separateurs_valides = []

for separateur in [";", ",", "\t", "|"]:
    colonnes = next(
        csv.reader([premiere_ligne], delimiter=separateur)
    )

    colonnes = [
        normaliser_nom_colonne(colonne)
        for colonne in colonnes
    ]

    if {"AN", "DEPCOM", "TYPEQU"}.issubset(colonnes):
        separateurs_valides.append(separateur)

if len(separateurs_valides) != 1:
    raise ValueError(
        "En-tête non reconnu : AN, DEPCOM et TYPEQU sont attendus."
    )

SEPARATEUR_BPE = separateurs_valides[0]

entete = pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    encoding="utf-8-sig",
    nrows=0,
)

noms = [normaliser_nom_colonne(c) for c in entete.columns]

if len(noms) != len(set(noms)):
    raise ValueError("Colonnes ambiguës après normalisation.")

print("Séparateur :", repr(SEPARATEUR_BPE))

Séparateur : ';'


In [7]:
#Extraire les observations par blocs

blocs_retenus = []
nombre_lignes_lues = 0
codes_inconnus = set()

with pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    dtype="string",
    encoding="utf-8-sig",
    keep_default_na=False,
    chunksize=20_000,
) as lecteur:

    for numero_bloc, bloc in enumerate(lecteur, start=1):
        bloc.columns = [
            normaliser_nom_colonne(c) for c in bloc.columns
        ]

        bloc["RANG_SOURCE_J10D"] = np.arange(
            nombre_lignes_lues + 1,
            nombre_lignes_lues + len(bloc) + 1,
        )

        nombre_lignes_lues += len(bloc)

        types = nettoyer_texte(bloc["TYPEQU"]).str.upper()
        communes = normaliser_code_commune(bloc["DEPCOM"])

        est_idf = communes.str[:2].isin(DEPARTEMENTS_IDF)
        est_domaine_sante = types.str.startswith("D", na=False)

        masque_inconnu = (
            est_idf
            & est_domaine_sante
            & ~types.isin(TYPES)
        )

        codes_inconnus.update(
            types.loc[masque_inconnu].dropna().tolist()
        )

        masque = est_idf & types.isin(TYPES)

        selection = bloc.loc[masque].copy()

        if not selection.empty:
            selection["TYPEQU_ANALYSE"] = types.loc[masque]
            selection["CODGEO_SOURCE"] = communes.loc[masque]
            blocs_retenus.append(selection)

        if numero_bloc % 25 == 0:
            print(f"{nombre_lignes_lues:,} lignes examinées")

if codes_inconnus:
    raise ValueError(
        "Codes du domaine D non prévus dans le référentiel : "
        + ", ".join(sorted(codes_inconnus))
    )

if not blocs_retenus:
    raise ValueError("Aucune observation retenue.")

sante = pd.concat(blocs_retenus, ignore_index=True)
del blocs_retenus

if not nettoyer_texte(sante["AN"]).eq("2025").fillna(False).all():
    raise ValueError("Millésime différent de 2025 dans l'extraction.")

print("Observations retenues :", len(sante))

500,000 lignes examinées
1,000,000 lignes examinées
1,500,000 lignes examinées
2,000,000 lignes examinées
2,500,000 lignes examinées
Observations retenues : 113073


In [8]:
# Ajouter la tracabilité

HASH_BPE = sha256_fichier(FICHIER_BPE)
DATE_TRAITEMENT = datetime.now(timezone.utc).isoformat()

sante["ID_OBSERVATION"] = (
    "BPE2025_"
    + HASH_BPE[:16]
    + "_"
    + sante["RANG_SOURCE_J10D"].astype("string")
)

sante["CATEGORIE_ANALYSE"] = sante["TYPEQU_ANALYSE"].map(TYPES)
sante["GROUPE_CODE"] = sante["TYPEQU_ANALYSE"].str[:2]
sante["GROUPE_ANALYSE"] = sante["GROUPE_CODE"].map(GROUPES)

sante["SOURCE_DONNEES"] = "INSEE_BPE_2025"
sante["STATUT_ACTIVITE_ACTUELLE"] = "NON_VERIFIE"

assert sante["ID_OBSERVATION"].is_unique
assert sante["GROUPE_ANALYSE"].notna().all()

display(sante["GROUPE_ANALYSE"].value_counts())

GROUPE_ANALYSE
PROFESSIONNELS_SANTE_RECENSES     79231
ENFANCE_FAMILLES_VIE_SOCIALE      16783
PHARMACIES_SERVICES_SANITAIRES     5246
STRUCTURES_SOINS                   3717
ACCOMPAGNEMENT_HANDICAP            2997
PROTECTION_HEBERGEMENT_SOCIAL      2580
SERVICES_PERSONNES_AGEES           2519
Name: count, dtype: int64

In [9]:
#Harmoniser les communes

sante["CODGEO"] = sante["CODGEO_SOURCE"]

arrondissements_paris = {
    f"751{numero:02d}" for numero in range(1, 21)
}

if codes_profil & arrondissements_paris:
    raise ValueError(
        "Le profil doit être communal, sans arrondissements parisiens."
    )

if "75056" in codes_profil:
    sante.loc[
        sante["CODGEO"].isin(arrondissements_paris),
        "CODGEO",
    ] = "75056"

if "93066" in codes_profil and "93059" not in codes_profil:
    sante.loc[
        sante["CODGEO"].eq("93059"),
        "CODGEO",
    ] = "93066"

hors_profil = sante.loc[
    ~sante["CODGEO"].isin(codes_profil)
].copy()

sauvegarder_csv(
    hors_profil,
    DOSSIER_INTERIM / "observations_hors_profil.csv",
)

print("Observations hors profil :", len(hors_profil))

if not hors_profil.empty:
    display(
        hors_profil[
            ["CODGEO_SOURCE", "CODGEO", "TYPEQU_ANALYSE"]
        ].drop_duplicates()
    )

    raise ValueError(
        "Résoudre les codes non appariés avant de poursuivre."
    )

Observations hors profil : 0


In [10]:
#Préparer les noms et adreesses

colonnes_facultatives = [
    "SIRET", "NOMRS", "CNOMRS",
    "NUMVOIE", "INDREP", "TYPVOIE", "LIBVOIE", "CADR",
    "LATITUDE", "LONGITUDE",
]

for colonne in colonnes_facultatives:
    if colonne not in sante.columns:
        sante[colonne] = pd.Series(
            pd.NA,
            index=sante.index,
            dtype="string",
        )

sante["NOM_AFFICHAGE"] = (
    nettoyer_texte(sante["NOMRS"])
    .fillna(nettoyer_texte(sante["CNOMRS"]))
)

parties_adresse = sante[
    ["NUMVOIE", "INDREP", "TYPVOIE", "LIBVOIE", "CADR"]
].apply(nettoyer_texte)

sante["ADRESSE_RECOMPOSEE"] = (
    parties_adresse.fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace("", pd.NA)
)

display(
    sante[
        [
            "CATEGORIE_ANALYSE",
            "NOM_AFFICHAGE",
            "ADRESSE_RECOMPOSEE",
            "CODGEO",
        ]
    ].head(10)
)

,CATEGORIE_ANALYSE,NOM_AFFICHAGE,ADRESSE_RECOMPOSEE,CODGEO
0,SOINS_COURTE_DUREE,SA CLINIQUE DU LOUVRE,17 RUE DES PRETRES SAINT GERMAIN L AUXERROIS,75056
1,CENTRES_SANTE,CDS DENTAIRE OPERA PYRAMIDES,13 AV DE L OPERA,75056
2,CENTRES_SANTE,CDS DENTAIRE DU CHATEAU RIVOLI,116 RUE RIVOLI,75056
3,CENTRES_SANTE,CDS DENTAIRE LES HALLES,100 RUE SAINT DENIS,75056
4,CENTRES_SANTE,CDS OPHTALMOLOGIQUE ET DENTAIRE,7 RUE TURBIGO,75056
5,CENTRES_SANTE,CDS MEDICO DENTAIRE CHATELET RIVOLI,9 BD SEBASTOPOL 9 11,75056
6,CENTRES_SANTE,CDS IWC QUAI DES ORFEVRES,42 QUAI DES ORFEVRES,75056
7,CENTRES_PREVENTION_MEDICALE,DISPENSAIRE ANTIVENERIEN DU SIDA DE LA CRF,43 RUE DE VALOIS,75056
8,CENTRES_PREVENTION_MEDICALE,CENTRE D EXAMENS DE SANTE,10 RUE DU COLONEL DRIANT,75056
9,MAISONS_SANTE_PLURIDISCIPLINAIRES,MSP DEVELOPPEMENT DES SOINS PRIMAIRE PARIS CENTRE,51 RUE JEAN JACQUES ROUSSEAU,75056


In [11]:
#Contrôler les SIRET

sante["SIRET_NETTOYE"] = nettoyer_texte(sante["SIRET"])

sante["SIRET_FORMAT_CONFORME"] = (
    sante["SIRET_NETTOYE"].str.fullmatch(r"\d{14}", na=False)
    & sante["SIRET_NETTOYE"].ne("0" * 14).fillna(False)
)

sante["SIRET_TYPE_REPETE"] = (
    sante["SIRET_FORMAT_CONFORME"]
    & sante.duplicated(
        subset=["SIRET_NETTOYE", "TYPEQU_ANALYSE"],
        keep=False,
    )
)

repetitions = sante.loc[
    sante["SIRET_TYPE_REPETE"]
].copy()

sauvegarder_csv(
    repetitions,
    DOSSIER_INTERIM / "siret_types_repetes_a_examiner.csv",
)

print("Observations avec SIRET et type répétés :", len(repetitions))
print("Aucune observation supprimée.")

Observations avec SIRET et type répétés : 8991
Aucune observation supprimée.


In [12]:
#Controler les coordonnées

sante["LATITUDE_NUM"] = convertir_nombre(sante["LATITUDE"])
sante["LONGITUDE_NUM"] = convertir_nombre(sante["LONGITUDE"])

coordonnees_presentes = (
    sante["LATITUDE_NUM"].notna()
    & sante["LONGITUDE_NUM"].notna()
)

coordonnees_plausibles = (
    sante["LATITUDE_NUM"].between(48.0, 49.3)
    & sante["LONGITUDE_NUM"].between(1.4, 3.7)
).fillna(False)

sante["STATUT_COORDONNEES"] = "ABSENTES_OU_NON_NUMERIQUES"

sante.loc[
    coordonnees_presentes,
    "STATUT_COORDONNEES",
] = "HORS_RECTANGLE_CONTROLE"

sante.loc[
    coordonnees_plausibles,
    "STATUT_COORDONNEES",
] = "PLAUSIBLES_A_VERIFIER"

controle_coordonnees = (
    sante.groupby(
        ["GROUPE_ANALYSE", "STATUT_COORDONNEES"]
    )
    .size()
    .reset_index(name="NB_OBSERVATIONS")
)

sauvegarder_csv(
    controle_coordonnees,
    DOSSIER_INTERIM / "controle_coordonnees.csv",
)

display(controle_coordonnees)

,GROUPE_ANALYSE,STATUT_COORDONNEES,NB_OBSERVATIONS
0,ACCOMPAGNEMENT_HANDICAP,ABSENTES_OU_NON_NUMERIQUES,1
1,ACCOMPAGNEMENT_HANDICAP,PLAUSIBLES_A_VERIFIER,2996
2,ENFANCE_FAMILLES_VIE_SOCIALE,PLAUSIBLES_A_VERIFIER,16783
3,PHARMACIES_SERVICES_SANITAIRES,ABSENTES_OU_NON_NUMERIQUES,23
4,PHARMACIES_SERVICES_SANITAIRES,PLAUSIBLES_A_VERIFIER,5223
5,PROFESSIONNELS_SANTE_RECENSES,ABSENTES_OU_NON_NUMERIQUES,18
6,PROFESSIONNELS_SANTE_RECENSES,PLAUSIBLES_A_VERIFIER,79213
7,PROTECTION_HEBERGEMENT_SOCIAL,ABSENTES_OU_NON_NUMERIQUES,2
8,PROTECTION_HEBERGEMENT_SOCIAL,PLAUSIBLES_A_VERIFIER,2578
9,SERVICES_PERSONNES_AGEES,ABSENTES_OU_NON_NUMERIQUES,1


In [13]:
#Compter par sous-catégorie

COLONNES_TYPES = {
    code: f"NB_OBS_{libelle}_BPE2025"
    for code, libelle in TYPES.items()
}

comptages = pd.crosstab(
    sante["CODGEO"],
    sante["TYPEQU_ANALYSE"],
)

comptages = (
    comptages.reindex(columns=list(TYPES), fill_value=0)
    .rename(columns=COLONNES_TYPES)
    .reset_index()
)

indicateurs = profil[["CODGEO"]].merge(
    comptages,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

colonnes_types = list(COLONNES_TYPES.values())

indicateurs[colonnes_types] = (
    indicateurs[colonnes_types]
    .fillna(0)
    .astype("int64")
)

display(indicateurs.head())

,CODGEO,NB_OBS_SOINS_COURTE_DUREE_BPE2025,NB_OBS_SOINS_SUITE_READAPTATION_BPE2025,NB_OBS_SOINS_LONGUE_DUREE_BPE2025,NB_OBS_ETABLISSEMENTS_PSYCHIATRIQUES_BPE2025,NB_OBS_CENTRES_CANCEROLOGIE_BPE2025,NB_OBS_SERVICES_URGENCES_BPE2025,NB_OBS_MATERNITES_BPE2025,NB_OBS_CENTRES_SANTE_BPE2025,NB_OBS_PSYCHIATRIE_AMBULATOIRE_BPE2025,NB_OBS_CENTRES_PREVENTION_MEDICALE_BPE2025,NB_OBS_SERVICES_DIALYSE_BPE2025,NB_OBS_HOSPITALISATION_DOMICILE_BPE2025,NB_OBS_MAISONS_SANTE_PLURIDISCIPLINAIRES_BPE2025,NB_OBS_SERVICES_ADDICTOLOGIE_BPE2025,NB_OBS_SANTE_MATERNELLE_INFANTILE_BPE2025,NB_OBS_PROFESSIONNELS_APPAREILLAGE_BPE2025,NB_OBS_AUDIOPROTHESISTES_BPE2025,NB_OBS_ERGOTHERAPEUTES_BPE2025,NB_OBS_PSYCHOMOTRICIENS_BPE2025,NB_OBS_DIETETICIENS_BPE2025,NB_OBS_PSYCHOLOGUES_BPE2025,NB_OBS_ALLERGOLOGUES_BPE2025,NB_OBS_ANESTHESISTES_REANIMATEURS_BPE2025,NB_OBS_CHIRURGIE_GENERALE_BPE2025,NB_OBS_CHIRURGIE_ORTHOPEDIQUE_PLASTIQUE_TRAUMATOLOGIQUE_BPE2025,NB_OBS_ENDOCRINOLOGUES_BPE2025,NB_OBS_GERIATRES_BPE2025,NB_OBS_HEMATOLOGUES_BPE2025,NB_OBS_MEDECINE_PHYSIQUE_READAPTATION_BPE2025,NB_OBS_NEUROLOGIE_BPE2025,NB_OBS_ONCOLOGIE_ANATOMIE_CYTOLOGIE_PATHOLOGIQUES_BPE2025,NB_OBS_RHUMATOLOGUES_BPE2025,NB_OBS_UROLOGIE_NEPHROLOGIE_BPE2025,NB_OBS_MEDECINS_GENERALISTES_BPE2025,NB_OBS_CARDIOLOGIE_BPE2025,NB_OBS_DERMATOLOGIE_VENEREOLOGIE_BPE2025,NB_OBS_GASTRO_ENTEROLOGIE_HEPATOLOGIE_BPE2025,NB_OBS_PSYCHIATRIE_BPE2025,NB_OBS_OPHTALMOLOGIE_BPE2025,NB_OBS_OTO_RHINO_LARYNGOLOGIE_BPE2025,NB_OBS_PEDIATRIE_BPE2025,NB_OBS_PNEUMOLOGIE_BPE2025,NB_OBS_RADIODIAGNOSTIC_IMAGERIE_MEDICALE_BPE2025,NB_OBS_STOMATOLOGIE_BPE2025,NB_OBS_GYNECOLOGIE_OBSTETRIQUE_BPE2025,NB_OBS_CHIRURGIENS_DENTISTES_BPE2025,NB_OBS_SAGES_FEMMES_BPE2025,NB_OBS_MASSEURS_KINESITHERAPEUTES_BPE2025,NB_OBS_PEDICURES_PODOLOGUES_BPE2025,NB_OBS_INFIRMIERS_BPE2025,NB_OBS_ORTHOPHONISTES_BPE2025,NB_OBS_ORTHOPTISTES_BPE2025,NB_OBS_LABORATOIRES_BIOLOGIE_MEDICALE_BPE2025,NB_OBS_AMBULANCES_BPE2025,NB_OBS_TRANSFUSION_SANGUINE_BPE2025,NB_OBS_ETABLISSEMENTS_THERMAUX_BPE2025,NB_OBS_PHARMACIES_BPE2025,NB_OBS_HEBERGEMENT_PERSONNES_AGEES_BPE2025,NB_OBS_SOINS_DOMICILE_PERSONNES_AGEES_BPE2025,NB_OBS_AIDE_PERSONNES_AGEES_BPE2025,NB_OBS_ACCUEIL_JEUNES_ENFANTS_BPE2025,NB_OBS_LIEUX_ACCUEIL_ENFANTS_PARENTS_BPE2025,NB_OBS_RELAIS_PETITE_ENFANCE_BPE2025,NB_OBS_ACCUEILS_LOISIRS_SANS_HEBERGEMENT_BPE2025,NB_OBS_CENTRES_SOCIAUX_BPE2025,NB_OBS_MEDIATION_FAMILIALE_BPE2025,NB_OBS_ESPACES_VIE_SOCIALE_BPE2025,NB_OBS_MICRO_CRECHES_FINANCEMENT_PAJE_BPE2025,NB_OBS_HEBERGEMENT_ENFANTS_HANDICAP_BPE2025,NB_OBS_SERVICES_ENFANTS_HANDICAP_BPE2025,NB_OBS_ACCUEIL_HEBERGEMENT_ADULTES_HANDICAP_BPE2025,NB_OBS_AIDE_ADULTES_HANDICAP_BPE2025,NB_OBS_TRAVAIL_PROTEGE_BPE2025,NB_OBS_SOINS_DOMICILE_ADULTES_HANDICAP_BPE2025,NB_OBS_AIDE_HANDICAP_VISUEL_AUDITIF_BPE2025,NB_OBS_HEBERGEMENT_PROTECTION_ENFANCE_BPE2025,NB_OBS_ACTION_EDUCATIVE_PROTECTION_ENFANCE_BPE2025,NB_OBS_HEBERGEMENT_REINSERTION_SOCIALE_BPE2025,NB_OBS_HEBERGEMENT_PROVISOIRE_BPE2025,NB_OBS_ACCUEIL_DEMANDEURS_ASILE_BPE2025,NB_OBS_AUTRES_HEBERGEMENTS_PUBLICS_DIFFICULTE_BPE2025,NB_OBS_AIDE_FEMMES_VULNERABILITE_BPE2025
0,75056,61,43,8,79,1,16,15,292,77,10,26,2,45,27,62,52,138,84,253,307,4493,29,239,102,672,89,19,11,51,148,131,136,132,2722,536,355,245,1302,512,189,240,71,1084,117,597,2446,326,4039,701,1556,999,177,166,149,13,0,864,228,28,205,833,31,19,1734,45,14,59,273,48,58,127,194,39,22,23,92,47,63,5,5,436,45
1,77001,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,77002,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,1,0,0,0,0,0,2,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0
3,77003,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,77004,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,

In [14]:
#Compter par groupe

COLONNES_GROUPES = {
    code: f"NB_OBS_{libelle}_BPE2025"
    for code, libelle in GROUPES.items()
}

for groupe, colonne_sortie in COLONNES_GROUPES.items():
    colonnes_a_sommer = [
        COLONNES_TYPES[code]
        for code in TYPES
        if code.startswith(groupe)
    ]

    indicateurs[colonne_sortie] = (
        indicateurs[colonnes_a_sommer].sum(axis=1)
    )

COLONNE_TOTAL = "NB_OBS_SANTE_ACTION_SOCIALE_BPE2025"

indicateurs[COLONNE_TOTAL] = (
    indicateurs[colonnes_types].sum(axis=1)
)

indicateurs["NB_TYPES_SANTE_ACTION_SOCIALE_PRESENTS"] = (
    indicateurs[colonnes_types].gt(0).sum(axis=1)
)

assert (
    indicateurs[list(COLONNES_GROUPES.values())].sum(axis=1)
    == indicateurs[COLONNE_TOTAL]
).all()

assert indicateurs[COLONNE_TOTAL].sum() == len(sante)

print("Comptages cohérents.")

Comptages cohérents.


In [15]:
#Calculer les parts

# Supprimer les anciennes parts J10d avant de les recalculer.
# Les comptages et les densités restent inchangés.

anciennes_parts = [
    colonne
    for colonne in indicateurs.columns
    if (
        colonne.startswith("PART_")
        and (
            colonne.endswith("_J10D_PCT")
            or colonne.endswith("_DANS_GROUPE_PCT")
        )
    )
]

indicateurs = indicateurs.drop(columns=anciennes_parts)

total = indicateurs[COLONNE_TOTAL].replace(0, np.nan)

colonnes_parts_groupes = {}
colonnes_parts_intragroupe = {}
series_parts = {}

# 1. Part de chaque groupe dans le total santé et action sociale.
for code, libelle in GROUPES.items():
    nom_part = f"PART_{libelle}_J10D_PCT"

    series_parts[nom_part] = (
        indicateurs[COLONNES_GROUPES[code]]
        .div(total)
        .mul(100)
    )

    colonnes_parts_groupes[code] = nom_part

# 2. Part de chaque sous-catégorie dans son groupe uniquement.
for code, libelle in TYPES.items():
    colonne_nombre = COLONNES_TYPES[code]
    colonne_groupe = COLONNES_GROUPES[code[:2]]

    nom_part = f"PART_{libelle}_DANS_GROUPE_PCT"

    series_parts[nom_part] = (
        indicateurs[colonne_nombre]
        .div(indicateurs[colonne_groupe].replace(0, np.nan))
        .mul(100)
    )

    colonnes_parts_intragroupe[code] = nom_part

indicateurs = pd.concat(
    [
        indicateurs,
        pd.DataFrame(series_parts, index=indicateurs.index),
    ],
    axis=1,
)

print("Parts de groupes :", len(colonnes_parts_groupes))
print("Parts intragroupe :", len(colonnes_parts_intragroupe))
print("Variables évitées :", len(TYPES))

Parts de groupes : 7
Parts intragroupe : 82
Variables évitées : 82


In [16]:
#Controler les parts

def verifier_parts(table, colonnes, colonne_total):
    avec_observations = table[colonne_total].gt(0)

    valeurs = table.loc[avec_observations, colonnes]

    assert valeurs.notna().all().all()
    assert ((valeurs >= 0) & (valeurs <= 100)).all().all()

    assert np.allclose(
        valeurs.sum(axis=1).to_numpy(dtype=float),
        100.0,
    )

    # Sans observation dans le périmètre :
    # les parts doivent rester manquantes.
    assert table.loc[
        ~avec_observations, colonnes
    ].isna().all().all()


# Les parts des groupes totalisent 100 %.
verifier_parts(
    indicateurs,
    list(colonnes_parts_groupes.values()),
    COLONNE_TOTAL,
)

# Les parts des sous-catégories totalisent 100 %
# séparément dans chaque groupe.
for groupe in GROUPES:
    colonnes = [
        colonnes_parts_intragroupe[code]
        for code in TYPES
        if code.startswith(groupe)
    ]

    verifier_parts(
        indicateurs,
        colonnes,
        COLONNES_GROUPES[groupe],
    )

print("Parts des groupes et parts intragroupe contrôlées.")

Parts des groupes et parts intragroupe contrôlées.


In [17]:
#Choisir la population

COLONNE_POPULATION = None
metadata_j10c = {}

if FICHIER_METADATA_J10C.is_file():
    with open(
        FICHIER_METADATA_J10C,
        "r",
        encoding="utf-8",
    ) as flux:
        metadata_j10c = json.load(flux)

if COLONNE_POPULATION is None:
    COLONNE_POPULATION = metadata_j10c.get("colonne_population")

if COLONNE_POPULATION is None:
    candidats = [
        colonne
        for colonne in [
            "POPULATION",
            "POPULATION_MUNICIPALE",
            "P23_POP",
            "P22_POP",
            "POP",
        ]
        if colonne in profil.columns
    ]

    if len(candidats) == 1:
        COLONNE_POPULATION = candidats[0]
    else:
        print("Candidats :", candidats)
        raise ValueError(
            "Renseigne COLONNE_POPULATION comme au J10c."
        )

if COLONNE_POPULATION not in profil.columns:
    raise ValueError("Colonne de population introuvable.")

population_par_code = (
    profil.set_index("CODGEO")[COLONNE_POPULATION]
    .pipe(convertir_nombre)
)

population = indicateurs["CODGEO"].map(population_par_code)

population_valide = population.gt(0).fillna(False)
denominateur = population.where(population_valide)

print("Population utilisée :", COLONNE_POPULATION)
print(
    "Communes sans population positive exploitable :",
    int((~population_valide).sum()),
)

Population utilisée : POPULATION_2022
Communes sans population positive exploitable : 0


In [18]:
#Calculer les densités

# Retirer les densités J10d déjà calculées.
anciennes_densites = [
    f"DENSITE_OBS_{libelle}_10000_HAB"
    for libelle in [*TYPES.values(), *GROUPES.values()]
]

indicateurs = indicateurs.drop(
    columns=anciennes_densites,
    errors="ignore",
)

# Conserver uniquement les densités des groupes.
DENSITES = {
    COLONNES_GROUPES[code]:
        f"DENSITE_OBS_{libelle}_10000_HAB"
    for code, libelle in GROUPES.items()
}

series_densites = {
    colonne_densite: (
        indicateurs[colonne_nombre]
        .div(denominateur)
        .mul(10_000)
    )
    for colonne_nombre, colonne_densite in DENSITES.items()
}

indicateurs = pd.concat(
    [
        indicateurs,
        pd.DataFrame(
            series_densites,
            index=indicateurs.index,
        ),
    ],
    axis=1,
)

print("Densités conservées :", len(DENSITES))
print("Densités de sous-catégories supprimées :", len(TYPES))

Densités conservées : 7
Densités de sous-catégories supprimées : 82


In [21]:
indicateurs.columns

Index(['CODGEO', 'NB_OBS_SOINS_COURTE_DUREE_BPE2025',
       'NB_OBS_SOINS_SUITE_READAPTATION_BPE2025',
       'NB_OBS_SOINS_LONGUE_DUREE_BPE2025',
       'NB_OBS_ETABLISSEMENTS_PSYCHIATRIQUES_BPE2025',
       'NB_OBS_CENTRES_CANCEROLOGIE_BPE2025',
       'NB_OBS_SERVICES_URGENCES_BPE2025', 'NB_OBS_MATERNITES_BPE2025',
       'NB_OBS_CENTRES_SANTE_BPE2025',
       'NB_OBS_PSYCHIATRIE_AMBULATOIRE_BPE2025',
       ...
       'PART_ACCUEIL_DEMANDEURS_ASILE_DANS_GROUPE_PCT',
       'PART_AUTRES_HEBERGEMENTS_PUBLICS_DIFFICULTE_DANS_GROUPE_PCT',
       'PART_AIDE_FEMMES_VULNERABILITE_DANS_GROUPE_PCT',
       'DENSITE_OBS_STRUCTURES_SOINS_10000_HAB',
       'DENSITE_OBS_PROFESSIONNELS_SANTE_RECENSES_10000_HAB',
       'DENSITE_OBS_PHARMACIES_SERVICES_SANITAIRES_10000_HAB',
       'DENSITE_OBS_SERVICES_PERSONNES_AGEES_10000_HAB',
       'DENSITE_OBS_ENFANCE_FAMILLES_VIE_SOCIALE_10000_HAB',
       'DENSITE_OBS_ACCOMPAGNEMENT_HANDICAP_10000_HAB',
       'DENSITE_OBS_PROTECTION_HEBERGEMENT_SOCIAL

In [19]:
#Joindre au niveau communal

nouvelles_colonnes = set(indicateurs.columns) - {"CODGEO"}
collisions = nouvelles_colonnes & set(profil.columns)

if collisions:
    raise ValueError(
        "Colonnes déjà présentes : "
        + ", ".join(sorted(collisions))
    )

if indicateurs.columns.duplicated().any():
    raise ValueError("Colonnes dupliquées dans les indicateurs.")

profil_j10d = profil.merge(
    indicateurs,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

assert len(profil_j10d) == len(profil)
assert profil_j10d["CODGEO"].is_unique

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j10d[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

print("Colonnes avant :", len(profil.columns))
print("Colonnes après :", len(profil_j10d.columns))
print("Indicateurs des journées précédentes conservés.")

Colonnes avant : 596
Colonnes après : 783
Indicateurs des journées précédentes conservés.


In [22]:
#Afficher les résultats

colonne_nom = next(
    (
        colonne
        for colonne in ["LIBELLE", "NOM_COMMUNE", "LIBGEO", "NOM_COM"]
        if colonne in profil_j10d.columns
    ),
    None,
)

colonnes_affichage = ["CODGEO"]

if colonne_nom:
    colonnes_affichage.append(colonne_nom)

colonnes_affichage += [
    COLONNE_POPULATION,
    "NB_OBS_MEDECINS_GENERALISTES_BPE2025",
    "DENSITE_OBS_PROFESSIONNELS_SANTE_RECENSES_10000_HAB",
    "NB_OBS_PHARMACIES_BPE2025",
    "NB_OBS_CENTRES_SANTE_BPE2025",
    "NB_OBS_HEBERGEMENT_PERSONNES_AGEES_BPE2025",
]

display(
    profil_j10d[colonnes_affichage]
    .sort_values(
        "NB_OBS_MEDECINS_GENERALISTES_BPE2025",
        ascending=False,
    )
    .head(20)
    .round(2)
)

print("Cergy et Pontoise :")

display(
    profil_j10d.loc[
        profil_j10d["CODGEO"].isin(["95127", "95500"]),
        colonnes_affichage,
    ].round(2)
)

,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_OBS_MEDECINS_GENERALISTES_BPE2025,DENSITE_OBS_PROFESSIONNELS_SANTE_RECENSES_10000_HAB,NB_OBS_PHARMACIES_BPE2025,NB_OBS_CENTRES_SANTE_BPE2025,NB_OBS_HEBERGEMENT_PERSONNES_AGEES_BPE2025
0,75056,Paris,2113705.0,2722,119.7,864,292,228
965,92012,Boulogne-Billancourt,120205.0,153,102.57,34,9,8
1070,94068,Saint-Maur-des-Fossés,76010.0,100,115.38,22,7,8
961,92002,Antony,64026.0,92,106.83,19,4,3
102,77108,Chelles,54372.0,83,60.14,15,7,5
985,92051,Neuilly-sur-Seine,59200.0,82,174.49,21,8,6
754,78646,Versailles,83918.0,81,106.65,25,6,6
1048,94028,Créteil,92859.0,72,53.74,23,9,6
1027,93066,Saint-Denis,148907.0,62,30.49,34,28,13
1016,93048,Montreuil,110758.0,62,51.37,28,19,7


Cergy et Pontoise :


,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_OBS_MEDECINS_GENERALISTES_BPE2025,DENSITE_OBS_PROFESSIONNELS_SANTE_RECENSES_10000_HAB,NB_OBS_PHARMACIES_BPE2025,NB_OBS_CENTRES_SANTE_BPE2025,NB_OBS_HEBERGEMENT_PERSONNES_AGEES_BPE2025
1119,95127,Cergy,69578.0,27,37.51,15,6,4
1220,95500,Pontoise,31623.0,19,62.61,8,1,1


In [23]:
bilan_categories = (
    sante.groupby(
        ["TYPEQU_ANALYSE", "CATEGORIE_ANALYSE", "GROUPE_ANALYSE"]
    )
    .size()
    .reset_index(name="NB_OBSERVATIONS")
    .sort_values("NB_OBSERVATIONS", ascending=False)
)

sauvegarder_csv(
    bilan_categories,
    DOSSIER_INTERIM / "bilan_categories.csv",
)

display(bilan_categories.head(30))

,TYPEQU_ANALYSE,CATEGORIE_ANALYSE,GROUPE_ANALYSE,NB_OBSERVATIONS
47,D279,MASSEURS_KINESITHERAPEUTES,PROFESSIONNELS_SANTE_RECENSES,12708
63,D505,ACCUEILS_LOISIRS_SANS_HEBERGEMENT,ENFANCE_FAMILLES_VIE_SOCIALE,10758
20,D250,PSYCHOLOGUES,PROFESSIONNELS_SANTE_RECENSES,9917
33,D265,MEDECINS_GENERALISTES,PROFESSIONNELS_SANTE_RECENSES,9277
49,D281,INFIRMIERS,PROFESSIONNELS_SANTE_RECENSES,8642
45,D277,CHIRURGIENS_DENTISTES,PROFESSIONNELS_SANTE_RECENSES,7744
42,D274,RADIODIAGNOSTIC_IMAGERIE_MEDICALE,PROFESSIONNELS_SANTE_RECENSES,4106
50,D282,ORTHOPHONISTES,PROFESSIONNELS_SANTE_RECENSES,3451
56,D307,PHARMACIES,PHARMACIES_SERVICES_SANITAIRES,3420
60,D502,ACCUEIL_JEUNES_ENFANTS,ENFANCE_FAMILLES_VIE_SOCIALE,3297


In [25]:
#Documenter chaque indicateur

lignes_dictionnaire = []

def documenter(colonne, description, unite, denominateur=""):
    lignes_dictionnaire.append({
        "COLONNE": colonne,
        "DESCRIPTION": description,
        "UNITE": unite,
        "DENOMINATEUR": denominateur,
    })


for code, libelle in TYPES.items():
    colonne_nombre = COLONNES_TYPES[code]

    documenter(
        colonne_nombre,
        f"Observations BPE du type {code} : {libelle}",
        "observations",
    )

    documenter(
        colonnes_parts_intragroupe[code],
        f"Part du type {code} dans son groupe",
        "%",
        COLONNES_GROUPES[code[:2]],
    )


for code, libelle in GROUPES.items():
    colonne_nombre = COLONNES_GROUPES[code]

    documenter(
        colonne_nombre,
        f"Observations du groupe {libelle}",
        "observations",
    )

    documenter(
        colonnes_parts_groupes[code],
        f"Part du groupe {libelle} dans J10d",
        "%",
        COLONNE_TOTAL,
    )

    documenter(
        DENSITES[colonne_nombre],
        f"Densité du groupe {libelle}",
        "observations pour 10 000 habitants",
        COLONNE_POPULATION,
    )


documenter(
    COLONNE_TOTAL,
    "Total des observations de santé et action sociale retenues",
    "observations",
)

documenter(
    "NB_TYPES_SANTE_ACTION_SOCIALE_PRESENTS",
    "Nombre de types BPE distincts présents",
    "types",
)

dictionnaire = pd.DataFrame(lignes_dictionnaire)

assert dictionnaire["COLONNE"].is_unique
assert set(dictionnaire["COLONNE"]) == (
    set(indicateurs.columns) - {"CODGEO"}
)

sauvegarder_csv(
    dictionnaire,
    DOSSIER_INTERIM / "dictionnaire_indicateurs_j10d.csv",
)

In [26]:
#Controles finaux

colonnes_effectifs = [
    *COLONNES_TYPES.values(),
    *COLONNES_GROUPES.values(),
    COLONNE_TOTAL,
    "NB_TYPES_SANTE_ACTION_SOCIALE_PRESENTS",
]

assert len(profil_j10d) == len(profil)
assert profil_j10d["CODGEO"].is_unique
assert set(profil_j10d["CODGEO"]) == codes_profil

assert sante["ID_OBSERVATION"].is_unique
assert sante["CODGEO"].isin(codes_profil).all()

assert indicateurs[colonnes_effectifs].notna().all().all()
assert indicateurs[colonnes_effectifs].ge(0).all().all()

assert (
    indicateurs[colonnes_types].sum(axis=1)
    == indicateurs[COLONNE_TOTAL]
).all()

assert indicateurs[COLONNE_TOTAL].sum() == len(sante)

for colonne in DENSITES.values():
    valeurs = indicateurs[colonne].dropna().astype(float)

    assert np.isfinite(valeurs).all()
    assert valeurs.ge(0).all()

    assert indicateurs.loc[
        ~population_valide, colonne
    ].isna().all()

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j10d[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

controle = pd.DataFrame([{
    "COMMUNES_ENTREE": len(profil),
    "COMMUNES_SORTIE": len(profil_j10d),
    "OBSERVATIONS_DETAIL": len(sante),
    "OBSERVATIONS_HORS_PROFIL": len(hors_profil),
    "OBSERVATIONS_SIRET_TYPE_REPETE": len(repetitions),
    "COMMUNES_SANS_POPULATION_VALIDE": int(
        (~population_valide).sum()
    ),
    "ETAT": (
        "CONTROLES_STRUCTURELS_OK"
        if repetitions.empty
        else "CONTROLES_STRUCTURELS_OK_REPETITIONS_A_EXAMINER"
    ),
}])

sauvegarder_csv(
    controle,
    DOSSIER_INTERIM / "controle_jointures.csv",
)

display(controle)

,COMMUNES_ENTREE,COMMUNES_SORTIE,OBSERVATIONS_DETAIL,OBSERVATIONS_HORS_PROFIL,OBSERVATIONS_SIRET_TYPE_REPETE,COMMUNES_SANS_POPULATION_VALIDE,ETAT
0,1266,1266,113073,0,8991,0,CONTROLES_STRUCTURELS_OK_REPETITIONS_A_EXAMINER


In [27]:
#Enregistrer les fichiers

sauvegarder_csv(sante, FICHIER_DETAIL)
sauvegarder_csv(indicateurs, FICHIER_COMMUNES)
sauvegarder_csv(profil_j10d, FICHIER_SORTIE)

metadonnees = {
    "notebook": "10d_sante_action_sociale_idf.ipynb",
    "date_traitement_utc": DATE_TRAITEMENT,
    "source": "INSEE_BPE_2025",
    "url_nomenclature": (
        "https://www.insee.fr/fr/metadonnees/source/fichier/"
        "BPE25_liste_hierarchisee_TYPEQU.html"
    ),
    "sha256_bpe": HASH_BPE,
    "fichier_bpe": str(FICHIER_BPE),
    "profil_entree": str(FICHIER_PROFIL),
    "sha256_profil_entree": sha256_fichier(FICHIER_PROFIL),
    "profil_sortie": str(FICHIER_SORTIE),
    "colonne_population": COLONNE_POPULATION,
    "types": TYPES,
    "groupes": GROUPES,
    "limites_specifiques": limites_specifiques,
    "nombre_observations": len(sante),
    "nombre_communes": len(profil_j10d),
    "observations_siret_type_repete": len(repetitions),
    "deduplication_automatique": False,
    "denominateur_parts_generales": COLONNE_TOTAL,
    "denominateur_parts_intragroupe": "Total du groupe de la catégorie",
    "limites": [
        "Observations BPE, pas de personnes ou bâtiments uniques.",
        "Champs et dates de référence variables selon les sources.",
        "Activité actuelle non vérifiée.",
        "Aucune mesure de fréquentation ou de disponibilité des soins.",
        "Densités rapportées à la population totale du profil.",
        "Capacités conservées brutes, non agrégées.",
        "Coordonnées contrôlées par rectangle uniquement.",
    ],
}

with open(
    DOSSIER_INTERIM / "metadata_j10d.json",
    "w",
    encoding="utf-8",
) as flux:
    json.dump(
        metadonnees,
        flux,
        ensure_ascii=False,
        indent=2,
    )

print("Détail :", FICHIER_DETAIL)
print("Indicateurs communaux :", FICHIER_COMMUNES)
print("Profil enrichi :", FICHIER_SORTIE)
print("J10d enregistré.")

Détail : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\sante_action_sociale_idf_j10d.csv
Indicateurs communaux : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j10d\sante_action_sociale_communes.csv
Profil enrichi : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10d.csv
J10d enregistré.
